# 04 - TreeSHAP

Explain the saved final XGBoost branch on the frozen test set. This notebook does not fit or retrain any XGBoost model. Fold-model OOF TreeSHAP is not available because fold model weights were not persisted; the saved OOF probabilities are not sufficient to recover feature attributions.

In [1]:
import hashlib
import json
import math
import platform
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import scipy
import shap
import sklearn
import xgboost as xgb
from scipy.stats import spearmanr
from sklearn.preprocessing import LabelEncoder

ROOT = Path(r"D:\skin_project")
OUT = ROOT / "outputs"
META = r"D:\D DRIVE DOWNLOADS\zr7vgbcyr2-1\metadata.csv"
SEED = 42
B_BACKGROUND = 200
CLASSES = ["ACK", "BCC", "MEL", "NEV", "SCC", "SEK"]
VARIABLES = ["grew", "itch", "changed", "elevation", "bleed", "hurt", "region", "age"]
FEATURES = VARIABLES
TARGET = "diagnostic"
np.random.seed(SEED)

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1048576), b""):
            digest.update(chunk)
    return digest.hexdigest()

versions = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit-learn": sklearn.__version__,
    "scipy": scipy.__version__,
    "xgboost": xgb.__version__,
    "shap": shap.__version__,
}
print("library versions:", json.dumps(versions, sort_keys=True))
assert shap.__version__ == "0.52.0"


c:\Users\Ramya Oli\.copilot\session-state\9f3be920-1523-47bb-bc4a-f28bd35ffddc\files\py313-shap52\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


library versions: {"numpy": "2.2.5", "pandas": "2.3.3", "python": "3.13.7", "scikit-learn": "1.6.1", "scipy": "1.15.3", "shap": "0.52.0", "xgboost": "3.2.0"}


In [2]:
base = np.load(OUT / "base_outputs_v1.npz", allow_pickle=True)
split = json.load(open(OUT / "split_v1.json", encoding="utf-8"))
y_train = base["y_train"].astype(int)
y_test = base["y_test"].astype(int)
assert base["classes"].astype(str).tolist() == CLASSES
assert (len(y_train), len(y_test)) == (1837, 461)

metadata = pd.read_csv(META)
metadata[TARGET] = metadata[TARGET].replace({"BOD": "SCC"})
metadata = metadata.set_index("img_id", drop=False)
train_df = metadata.loc[split["train_img_ids"]].reset_index(drop=True)
test_df = metadata.loc[split["test_img_ids"]].reset_index(drop=True)
label_encoder = LabelEncoder().fit(train_df[TARGET])
assert label_encoder.classes_.astype(str).tolist() == CLASSES
assert np.array_equal(label_encoder.transform(train_df[TARGET]), y_train)
assert np.array_equal(label_encoder.transform(test_df[TARGET]), y_test)

artifact_path = OUT / "xgb_final_v1.joblib"
artifact = joblib.load(artifact_path)
model = artifact["model"]
model_columns = artifact["columns"]
X_train = pd.get_dummies(train_df[FEATURES], dummy_na=True).astype(float)
X_train = X_train.reindex(columns=model_columns)
X_test = pd.get_dummies(test_df[FEATURES], dummy_na=True).reindex(
    columns=model_columns, fill_value=0
).astype(float)
assert X_train.shape == (1837, 40)
assert X_test.shape == (461, 40)

saved_test_probability = base["test_xgb"]
recomputed_test_probability = model.predict_proba(X_test)
reproduction_error = float(np.max(np.abs(saved_test_probability - recomputed_test_probability)))
assert reproduction_error < 1e-6, reproduction_error

print("train/test:", X_train.shape, X_test.shape)
print("final XGB sha256:", sha256(artifact_path))
print("test XGB prediction reproduction max abs error:", reproduction_error)


train/test: (1837, 40) (461, 40)
final XGB sha256: 3000cdcc1b10d1087f703258ab6e6fa3737c753c6706b482df9da1267c42df97
test XGB prediction reproduction max abs error: 0.0


In [3]:
def stratified_background(labels, n, seed):
    rng = np.random.default_rng(seed)
    pool = np.arange(len(labels))
    present = np.unique(labels)
    quotas = np.full(len(present), n // len(present), dtype=int)
    quotas[:n % len(present)] += 1
    selected = []
    for cls, quota in zip(present, quotas):
        candidates = pool[labels == cls]
        take = min(int(quota), len(candidates))
        if take:
            selected.extend(rng.choice(candidates, size=take, replace=False).tolist())
    selected = np.asarray(selected, dtype=int)
    if len(selected) < n:
        remaining = np.setdiff1d(pool, selected)
        selected = np.concatenate([
            selected,
            rng.choice(remaining, size=n-len(selected), replace=False),
        ])
    selected = np.sort(selected)
    assert len(selected) == n
    assert len(np.unique(selected)) == n
    assert np.isin(selected, pool).all()
    return selected

background_indices = stratified_background(y_train, B_BACKGROUND, SEED)
background = X_train.iloc[background_indices].copy()
assert len(background) == 200
assert set(np.unique(y_train[background_indices])) == set(range(6))

column_to_variable = []
for column in model_columns:
    matches = [
        i for i, variable in enumerate(VARIABLES)
        if column == variable or str(column).startswith(variable + "_")
    ]
    assert len(matches) == 1, (column, matches)
    column_to_variable.append(matches[0])
column_to_variable = np.asarray(column_to_variable, dtype=int)
group_columns = [
    np.flatnonzero(column_to_variable == i)
    for i in range(len(VARIABLES))
]


In [4]:
def normalize_tree_shap(values, n_rows):
    values = np.asarray(values, dtype=float)
    if values.ndim == 2:
        values = values[:, :, None]
    if values.shape == (n_rows, len(model_columns), len(CLASSES)):
        return values
    if values.shape == (len(model_columns), len(CLASSES), n_rows):
        return np.transpose(values, (2, 0, 1))
    raise AssertionError("Unexpected TreeSHAP shape: " + str(values.shape))

def group_feature_shap(values):
    grouped = np.zeros((values.shape[0], len(CLASSES), len(VARIABLES)), dtype=float)
    for variable_index, columns in enumerate(group_columns):
        grouped[:, :, variable_index] = values[:, columns, :].sum(axis=1)
    assert np.allclose(grouped.sum(axis=2), values.sum(axis=1), atol=1e-8)
    return grouped

def shapley_from_coalition_values(coalition_values):
    coalition_values = np.asarray(coalition_values, dtype=float)
    assert coalition_values.shape == (256, len(CLASSES))
    phi = np.zeros((8, len(CLASSES)), dtype=float)
    denominator = math.factorial(8)
    for variable_index in range(8):
        bit = 1 << variable_index
        for coalition in range(256):
            if coalition & bit:
                continue
            size = coalition.bit_count()
            weight = (
                math.factorial(size)
                * math.factorial(8 - size - 1)
                / denominator
            )
            phi[variable_index] += weight * (
                coalition_values[coalition | bit] - coalition_values[coalition]
            )
    return phi

# Unit test: an additive 8-variable, 6-output game must recover each variable's
# exact contribution and satisfy the efficiency identity.
toy_phi = np.arange(1, 49, dtype=float).reshape(8, 6) / 10
toy_base = np.arange(6, dtype=float) / 7
toy_values = np.empty((256, 6), dtype=float)
for coalition in range(256):
    present = np.asarray([(coalition >> i) & 1 for i in range(8)], dtype=float)
    toy_values[coalition] = toy_base + present @ toy_phi
toy_result = shapley_from_coalition_values(toy_values)
assert np.allclose(toy_result, toy_phi, atol=1e-12)
assert np.allclose(toy_result.sum(axis=0), toy_values[-1] - toy_values[0], atol=1e-12)
print("toy additive grouped-Shapley unit test: PASS")


toy additive grouped-Shapley unit test: PASS


In [5]:
# Raw/log-odds TreeSHAP for the final model on all test rows.
raw_masker = shap.maskers.Independent(
    background.to_numpy(dtype=np.float64),
    max_samples=B_BACKGROUND,
)
raw_explainer = shap.TreeExplainer(
    model,
    data=raw_masker,
    feature_perturbation="interventional",
    model_output="raw",
)
raw_values = normalize_tree_shap(
    raw_explainer.shap_values(X_test, check_additivity=True),
    len(X_test),
)
raw_expected = np.asarray(raw_explainer.expected_value, dtype=float).reshape(-1)
assert raw_expected.shape == (6,)
test_margin = model.predict(X_test, output_margin=True)
raw_residual = test_margin - (raw_values.sum(axis=1) + raw_expected[None, :])
raw_max_error = float(np.max(np.abs(raw_residual)))
assert raw_max_error < 1e-4, raw_max_error
raw_grouped = group_feature_shap(raw_values)

print("raw additivity max abs residual:", raw_max_error)
print("raw residual diagnostics by class:")
for class_index, class_name in enumerate(CLASSES):
    residual = raw_residual[:, class_index]
    is_constant_offset = float(np.ptp(residual)) < 1e-4
    print(
        f"  {class_name}: mean={residual.mean():.8g}, "
        f"std={residual.std():.8g}, max_abs={np.abs(residual).max():.8g}, "
        f"constant_offset_within_1e-4={is_constant_offset}"
    )


100%|===================| 2753/2766 [02:05<00:00]        

raw additivity max abs residual: 4.293333836002944e-06
raw residual diagnostics by class:
  ACK: mean=-1.4873668e-08, std=6.8594047e-07, max_abs=4.1107408e-06, constant_offset_within_1e-4=True
  BCC: mean=-3.5798834e-08, std=4.3090157e-07, max_abs=2.0387125e-06, constant_offset_within_1e-4=True
  MEL: mean=-4.7846278e-07, std=1.4156409e-06, max_abs=4.2933338e-06, constant_offset_within_1e-4=True
  NEV: mean=-3.6313653e-07, std=1.1900931e-06, max_abs=3.8695735e-06, constant_offset_within_1e-4=True
  SCC: mean=-9.1635643e-08, std=5.9097939e-07, max_abs=2.5212893e-06, constant_offset_within_1e-4=True
  SEK: mean=7.3477158e-08, std=4.6232715e-07, max_abs=1.9709087e-06, constant_offset_within_1e-4=True


In [6]:
def exact_probability_coalitions(model, cases, background, batch_cases=2):
    n_cases = len(cases)
    n_background = len(background)
    n_features = len(model_columns)
    values = np.empty((n_cases, 256, len(CLASSES)), dtype=np.float64)
    background_array = background.to_numpy(dtype=np.float32)
    case_array = cases.to_numpy(dtype=np.float32)
    coalition_matrix = np.empty(
        (batch_cases * 256 * n_background, n_features), dtype=np.float32
    )

    start = time.perf_counter()
    first_batch_seconds = None
    for batch_start in range(0, n_cases, batch_cases):
        current_cases = case_array[batch_start:batch_start + batch_cases]
        current_count = len(current_cases)
        rows_per_case = 256 * n_background
        for local_index, case_row in enumerate(current_cases):
            case_matrix = np.broadcast_to(
                background_array[None, :, :],
                (256, n_background, n_features),
            ).copy()
            for coalition in range(256):
                for variable_index, columns in enumerate(group_columns):
                    if (coalition >> variable_index) & 1:
                        case_matrix[coalition][:, columns] = case_row[columns]
            begin = local_index * rows_per_case
            coalition_matrix[begin:begin + rows_per_case] = case_matrix.reshape(
                rows_per_case, n_features
            )

        used_rows = current_count * rows_per_case
        coalition_frame = pd.DataFrame(
            coalition_matrix[:used_rows],
            columns=model_columns,
        )
        probabilities = model.predict_proba(coalition_frame)
        probabilities = np.asarray(probabilities, dtype=np.float64).reshape(
            current_count, 256, n_background, len(CLASSES)
        )
        values[batch_start:batch_start + current_count] = probabilities.mean(axis=2)
        if first_batch_seconds is None:
            first_batch_seconds = time.perf_counter() - start
            print(
                f"exact probability coalition batch ({current_count} cases): "
                f"{first_batch_seconds:.2f}s; estimated full test runtime: "
                f"{first_batch_seconds * n_cases / current_count / 60:.1f}min"
            )
    return values, first_batch_seconds

test_rows = X_test.iloc[:2]
benchmark_values, benchmark_seconds = exact_probability_coalitions(
    model, test_rows, background, batch_cases=2
)
assert benchmark_values.shape == (2, 256, 6)


exact probability coalition batch (2 cases): 0.21s; estimated full test runtime: 0.0min


In [7]:
probability_start = time.perf_counter()
probability_coalition_values, full_batch_seconds = exact_probability_coalitions(
    model, X_test, background, batch_cases=2
)
probability_grouped = np.empty((len(X_test), 8, 6), dtype=np.float64)
for row_index in range(len(X_test)):
    probability_grouped[row_index] = shapley_from_coalition_values(
        probability_coalition_values[row_index]
    )

empty_probability = probability_coalition_values[:, 0, :]
full_probability = probability_coalition_values[:, -1, :]
test_probability = model.predict_proba(X_test)
assert np.allclose(full_probability, test_probability, atol=1e-6)
probability_efficiency_error = float(np.max(np.abs(
    probability_grouped.sum(axis=1) - (full_probability - empty_probability)
)))
assert probability_efficiency_error < 1e-5, probability_efficiency_error

test_prediction = test_probability.argmax(axis=1)
probability_predicted_slice = np.asarray([
    probability_grouped[i, :, test_prediction[i]]
    for i in range(len(X_test))
])
raw_predicted_slice = np.asarray([
    raw_grouped[i, test_prediction[i], :]
    for i in range(len(X_test))
])

print("probability grouped-Shapley efficiency max abs error:", probability_efficiency_error)
print("probability coalition runtime minutes:", round((time.perf_counter()-probability_start)/60, 2))


exact probability coalition batch (2 cases): 0.23s; estimated full test runtime: 0.9min
probability grouped-Shapley efficiency max abs error: 6.661338147750939e-16
probability coalition runtime minutes: 0.78


In [8]:
# Shortcut audit: report average absolute predicted-class attribution for
# rows activating an unknown/NaN dummy versus rows with known values.
unknown_audit = {}
for variable_index, variable in enumerate(VARIABLES):
    unknown_columns_for_variable = [
        column for column in model_columns
        if str(column).startswith(variable + "_")
        and any(token in str(column).upper() for token in ("_UNK", "_NAN"))
    ]
    if unknown_columns_for_variable:
        unknown_mask = (
            X_test[unknown_columns_for_variable].sum(axis=1).to_numpy() > 0
        )
    else:
        unknown_mask = np.zeros(len(X_test), dtype=bool)
    known_mask = ~unknown_mask
    cls = test_prediction
    predicted_phi = np.asarray([
        probability_grouped[i, variable_index, cls[i]]
        for i in range(len(X_test))
    ])
    unknown_audit[variable] = {
        "unknown_count": int(unknown_mask.sum()),
        "known_count": int(known_mask.sum()),
        "unknown_mean_abs_phi": (
            float(np.mean(np.abs(predicted_phi[unknown_mask])))
            if unknown_mask.any() else None
        ),
        "known_mean_abs_phi": (
            float(np.mean(np.abs(predicted_phi[known_mask])))
            if known_mask.any() else None
        ),
    }

print("shortcut audit: mean |predicted-class probability phi|, unknown vs known")
for variable, values in unknown_audit.items():
    print(variable, values)


shortcut audit: mean |predicted-class probability phi|, unknown vs known
grew {'unknown_count': 104, 'known_count': 357, 'unknown_mean_abs_phi': 0.05926154072386959, 'known_mean_abs_phi': 0.09953050045326575}
itch {'unknown_count': 1, 'known_count': 460, 'unknown_mean_abs_phi': 0.13297009137584304, 'known_mean_abs_phi': 0.08271328874332162}
changed {'unknown_count': 102, 'known_count': 359, 'unknown_mean_abs_phi': 0.16569269978328052, 'known_mean_abs_phi': 0.060417636438988134}
elevation {'unknown_count': 0, 'known_count': 461, 'unknown_mean_abs_phi': None, 'known_mean_abs_phi': 0.10662365638390098}
bleed {'unknown_count': 1, 'known_count': 460, 'unknown_mean_abs_phi': 0.20508150630585265, 'known_mean_abs_phi': 0.0822288553981749}
hurt {'unknown_count': 2, 'known_count': 459, 'unknown_mean_abs_phi': 0.11621484680130262, 'known_mean_abs_phi': 0.02745583665008469}
region {'unknown_count': 0, 'known_count': 461, 'unknown_mean_abs_phi': None, 'known_mean_abs_phi': 0.13664367059869612}
age 

In [9]:
# Native XGBoost pred_contribs is a rank-correlation cross-check only.
native = np.asarray(
    model.get_booster().predict(
        xgb.DMatrix(X_test.iloc[:20], feature_names=model_columns),
        pred_contribs=True,
    ),
    dtype=float,
)
assert native.shape == (20, len(CLASSES), len(model_columns) + 1), native.shape
native_feature_values = native[:, :, :-1]
native_grouped = np.zeros((20, len(CLASSES), len(VARIABLES)), dtype=float)
for variable_index, columns in enumerate(group_columns):
    native_grouped[:, :, variable_index] = native_feature_values[:, :, columns].sum(axis=2)
native_rank_correlations = [
    spearmanr(
        np.mean(np.abs(native_grouped[i]), axis=0),
        np.mean(np.abs(raw_grouped[i]), axis=0),
    ).statistic
    for i in range(20)
]
native_rank_spearman = float(np.nanmean(native_rank_correlations))
print("native pred_contribs vs exact probability grouped rank Spearman:", native_rank_spearman)

global_importance_by_class = np.mean(np.abs(probability_grouped), axis=0).T
print("global mean |probability SHAP| by class and variable")
for class_index, class_name in enumerate(CLASSES):
    counts = int((y_test == class_index).sum())
    print(class_name, "n=", counts, dict(zip(
        VARIABLES,
        np.round(global_importance_by_class[class_index], 6),
    )))


native pred_contribs vs exact probability grouped rank Spearman: 0.9190476190476191
global mean |probability SHAP| by class and variable
ACK n= 127 {'grew': np.float64(0.091652), 'itch': np.float64(0.080354), 'changed': np.float64(0.082389), 'elevation': np.float64(0.106022), 'bleed': np.float64(0.022789), 'hurt': np.float64(0.012782), 'region': np.float64(0.09322), 'age': np.float64(0.061059)}
BCC n= 174 {'grew': np.float64(0.05239), 'itch': np.float64(0.037693), 'changed': np.float64(0.056717), 'elevation': np.float64(0.097495), 'bleed': np.float64(0.10187), 'hurt': np.float64(0.027657), 'region': np.float64(0.113647), 'age': np.float64(0.080001)}
MEL n= 7 {'grew': np.float64(0.024909), 'itch': np.float64(0.029454), 'changed': np.float64(0.050573), 'elevation': np.float64(0.036851), 'bleed': np.float64(0.010109), 'hurt': np.float64(0.004623), 'region': np.float64(0.020074), 'age': np.float64(0.027275)}
NEV n= 49 {'grew': np.float64(0.01686), 'itch': np.float64(0.053268), 'changed': n

In [10]:
np.savez(
    OUT / "treeshap_v1.npz",
    classes=np.asarray(CLASSES),
    variables=np.asarray(VARIABLES),
    test_row_keys=np.asarray(split["test_img_ids"], dtype=str),
    test_true_y=y_test,
    test_predicted_class=test_prediction,
    test_full_raw=raw_grouped,
    test_full_probability=np.transpose(probability_grouped, (0, 2, 1)),
    test_grouped_raw_predicted=raw_predicted_slice,
    test_grouped_probability_predicted=probability_predicted_slice,
    test_raw_expected_values=raw_expected,
    test_probability_expected_values=empty_probability,
    test_background_indices=background_indices,
    test_raw_additivity_residual=raw_residual,
    test_raw_additivity_max_abs=raw_max_error,
    test_probability_efficiency_max_abs=probability_efficiency_error,
    native_pred_contrib_rank_spearman=native_rank_spearman,
    exact_method=np.asarray("grouped_interventional_256_coalitions"),
    oof_status=np.asarray("not_computed: fold model weights are not persisted; no retraining"),
    library_versions=np.asarray(json.dumps(versions, sort_keys=True)),
)

with np.load(OUT / "treeshap_v1.npz", allow_pickle=False) as saved:
    assert saved["test_full_raw"].shape == (461, 6, 8)
    assert saved["test_full_probability"].shape == (461, 6, 8)
    assert saved["test_grouped_raw_predicted"].shape == (461, 8)
    assert saved["test_grouped_probability_predicted"].shape == (461, 8)
    assert np.isfinite(saved["test_full_raw"]).all()
    assert np.isfinite(saved["test_full_probability"]).all()
    assert saved["test_row_keys"].tolist() == split["test_img_ids"]
print("saved", OUT / "treeshap_v1.npz")


saved D:\skin_project\outputs\treeshap_v1.npz
